# Erg-synced stroke detection (Concept2 PM5 + IMU)

Alternative to `stroke_detection.ipynb`: the **PM5 stroke log decides which strokes happened** (and gives power / pace / rate for each), and the **simultaneous IMU gives exactly when** each one starts. Each stroke is then binned into 30 bins (Slade) as before.

**Why not just use the PM5 times?** The logbook export rounds stroke timestamps to 0.5 s ticks (intervals read 1.5 / 2.0 s at ~36 spm), so a single stroke time is only good to ±0.25 s. The first few strokes (start sequence, flywheel spin-up) are also unreliable: no stroke rate, 0.5 s apart, 66 spm.

**Sync logic** (`erg_stroke_sync.py`):
1. **Start anchor**: the PM5 timer starts on the first pull, so erg t = 0 ↔ the rower's first movement in the IMU after sitting still.
2. **Fine offset**: the PM logs each stroke a roughly constant delay after its catch. One offset is fitted so all *reliable* erg strokes line up with IMU catches.
3. **Per-stroke catch**: each erg stroke's catch is found in the IMU within ±0.4 s of its predicted time. Offset + clock drift are refitted and the search is repeated.
4. **Stroke k = IMU catch k → IMU catch k+1**, labelled with erg stroke k's watts / spm / pace.

**Recording protocol this relies on**
- Start IMU logging *before* rowing, then sit still at the catch for ≥ 5 s.
- Start rowing from rest (Just Row / a fresh piece), so the PM5 timer starts with the first pull.
- If that's not possible, put the IMU time of the PM's first pull in `OFFSET_S` (e.g. from a video or a sharp tap).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

import imu_preprocessing as ip
import stroke_detection as sd
import erg_stroke_sync as es

ERG_PATH = Path("../test data/erg/concept2-result-115770245.csv")
IMU_PATH = None          # cleaned IMU recorded simultaneously with ERG_PATH (imu_clean/...)
OFFSET_S = None          # manual sync: IMU time (s) when the PM5 timer read 0; None = onset
SKIP_FIRST = 5           # erg strokes at the start treated as unreliable
N_BINS, BIN_METHOD = 30, "resample"
OUT_DIR = Path("../test data/strokes_synced")

## 1. PM5 stroke log + reliability flags
Unreliable strokes are kept as stroke *boundaries* but not output as strokes.

In [ ]:
erg_real = es.load_c2_strokes(ERG_PATH, skip_first=SKIP_FIRST)
print(f"{len(erg_real)} strokes, {erg_real.reliable.sum()} reliable")
print("PM timestamp intervals (s):", erg_real.t_erg.diff().round(1).value_counts().sort_index().to_dict())
erg_real.head(10)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 4.5), sharex=True)
bad = ~erg_real.reliable
for ax, col, unit in zip(axes, ["spm", "watts"], ["spm", "W"]):
    ax.plot(erg_real.t_erg, erg_real[col], ".-", lw=0.6, ms=3)
    ax.plot(erg_real.t_erg[bad], erg_real[col][bad], "x", color="C3", label="unreliable")
    ax.set_ylabel(unit)
axes[0].legend(fontsize=8); axes[-1].set_xlabel("PM5 time (s)")
plt.tight_layout()

## 2. Paired IMU + erg data
The IMU file currently in `test data/imu` was **not** recorded with this erg piece, so until a simultaneous pair exists the notebook uses a synthetic pair:
a simulated thigh IMU (8 s still lead-in, ~36 → 40 spm) and a PM5 log generated from its true catches. The log is delayed 0.8 s, rounded to 0.5 s ticks, and has a garbage start sequence.

Set `IMU_PATH` above to switch to real data.

In [ ]:
if IMU_PATH is not None:
    clean = pd.read_csv(IMU_PATH)
    erg = erg_real
    true_catches = None
else:
    sim, true_catches = sd.simulate_erg_imu(duration_s=len(erg_real) * 1.65 + 10, spm=(36, 40),
                                            lead_s=8, seed=0)
    clean, _ = ip.clean_imu(sim)
    log = es.simulate_pm5_log(true_catches, onset_s=true_catches[0], log_delay_s=0.8)
    tmp = OUT_DIR / "_synthetic_pm5.csv"; OUT_DIR.mkdir(parents=True, exist_ok=True)
    log.to_csv(tmp, index=False)
    erg = es.load_c2_strokes(tmp, skip_first=SKIP_FIRST)

strokes, diag = es.sync_strokes(clean, erg, offset_s=OFFSET_S)
diag

Check the diagnostics:
- `start_count_check` — the erg and IMU stroke counts before the first reliable stroke should match.
- `resid_std_s` ≈ 0.14 s — just the PM's 0.5 s rounding; much larger means a bad sync.
- `reliable_matched` close to 100 %.
- No `warning`.

In [ ]:
x = sd.detect_strokes(clean, return_signal=True)[1]
t = clean.t_s.to_numpy()
fig, axes = plt.subplots(2, 1, figsize=(11, 6))
for ax, (t0, t1), title in zip(axes, [(diag["tau0_s"] - 3, diag["tau0_s"] + 15), (t[-1] - 15, t[-1])],
                               ["Start: PM5 timer zero = IMU movement onset", "End of piece"]):
    m = (t >= t0) & (t <= t1)
    ax.plot(t[m], x[m], color="k", lw=0.8, label="IMU detection signal (drive +)")
    ax.axvline(diag["tau0_s"], color="C1", lw=1.5, label="onset / PM t=0")
    e = erg.t_erg + diag["tau0_s"]
    ax.plot(e, np.full(len(e), x.max() * 1.1), "v", color="C0", ms=5, label="PM5 log time")
    s = strokes[(strokes.t_start >= t0) & (strokes.t_start <= t1)]
    ax.plot(s.t_start[s.valid], x[s.start_idx[s.valid]], "o", color="C2", label="synced catch (valid)")
    ax.plot(s.t_start[~s.valid], x[s.start_idx[~s.valid]], "o", color="C3", label="catch (unreliable erg stroke)")
    for _, r in s.iterrows():
        ax.annotate(int(r.erg_stroke), (r.t_start, x[int(r.start_idx)]), fontsize=7,
                    xytext=(0, -12), textcoords="offset points", ha="center")
    if true_catches is not None:
        ax.vlines(true_catches, x.min(), x.max(), color="0.6", ls="--", lw=0.6, label="true catch (sim)")
    ax.set_xlim(t0, t1); ax.set_title(title, fontsize=10)
axes[0].legend(fontsize=7, loc="upper right", ncol=3); axes[-1].set_xlabel("IMU time (s)")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
v = strokes[strokes.valid]
axes[0].plot(v.t_start, v.catch_resid_s * 1e3, ".", ms=3)
axes[0].set_ylabel("IMU catch − predicted (ms)"); axes[0].set_xlabel("IMU time (s)")
axes[0].set_title("Should be flat noise within ±250 ms (PM tick)", fontsize=9)
axes[1].plot(60 / v.spm, v.duration_s, ".", ms=3)
lim = [v.duration_s.min(), v.duration_s.max()]; axes[1].plot(lim, lim, "k--", lw=0.8)
axes[1].set_xlabel("PM5 60/spm (s)"); axes[1].set_ylabel("IMU stroke duration (s)")
plt.tight_layout()
if true_catches is not None:
    err = v.t_start.values - true_catches[v.erg_stroke.astype(int).values - 1]
    print(f"synthetic check: catch error {1e3*err.mean():+.1f} ± {1e3*err.std():.1f} ms, "
          f"labels correct {np.mean(abs(err) < 0.1):.0%}")

## 3. Bin each synced stroke into 30 bins + attach erg targets

In [ ]:
WEIGHT_KG, HEIGHT_M = None, None
binned, table = es.synced_features(clean, strokes, n_bins=N_BINS, method=BIN_METHOD,
                                   weight=WEIGHT_KG, height=HEIGHT_M)
print("binned:", binned.shape, " table:", table.shape)

pct = np.arange(N_BINS) / N_BINS * 100
fig, axes = plt.subplots(2, 3, figsize=(12, 5.5), sharex=True)
for j, (ax, c) in enumerate(zip(axes.flat, ip.SIGNALS)):
    ax.plot(pct, binned[:, :, j].T, color="0.75", lw=0.4)
    ax.plot(pct, binned[:, :, j].mean(0), color="C0", lw=2); ax.set_title(c, fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel("% stroke (catch → catch)")
plt.tight_layout()
table.head()

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
stem = Path(IMU_PATH).stem if IMU_PATH is not None else "synthetic"
strokes.to_csv(OUT_DIR / f"{stem}_synced_strokes.csv", index=False)
table.to_csv(OUT_DIR / f"{stem}_synced_bins{N_BINS}.csv", index=False)
sorted(p.name for p in OUT_DIR.iterdir())